# Self-recognition: Qwen3.5-27B vs its SGTR fine-tunes

Runs the CHIVE self-recognition experiment on four models, all served from one vLLM server on this GPU:

| Name | Model |
|---|---|
| `base` | Qwen/Qwen3.5-27B |
| `sgtr0`, `sgtr1`, `sgtr2` | praxisresearch/hf_qwen35_27b_sgtr_syspopped_{0,1,2}: LoRA adapters trained (seeds 0–2) to pick Qwen3.5-27B's summary over Claude 2.1's |

**Articles are held out.** The adapters were trained on 999 of the 1,000 XSum articles in sgtr-em, so this uses its CNN articles (with their existing Claude 2.1 summaries), none of which were in training. "Own summary" = the base model's summary at temperature 0, as in the adapters' training data.

**What runs**
1. Prompts for `N_ARTICLES` CNN articles. By default this reuses the committed `introspection/self_recognition/cnn_6.jsonl`, the exact prompts behind the reported results; set `REUSE_PROMPTS = False` to build new ones.
2. **Position test**: every model × both answer formats (`syspopped` = the trained "1 or 2 only" format, no system prompt; `explain` = our format) × own summary first and second × `N_POSITION` samples.
3. **CHIVE investigations**: every model × every article, with prewritten screening.
4. Zip the results for download.

**Resume mode.** To continue an earlier run instead (for example one stopped partway), unzip its downloaded results folder into the repo, point `PREVIOUS` at it and set `RESUME = True`. The notebook then skips the position test, keeps the finished investigations, runs only those listed in `NEW_INVESTIGATIONS`, and verifies them all.

**Before you start**
- *Runtime → Change runtime type* → **G4 GPU** (96 GB) or **H100 GPU** (80 GB). An A100 works only if it is the 80 GB version; the usual 40 GB A100, L4 and T4 are too small for this model in bf16. High-RAM is not needed.
- In the 🔑 **Secrets** panel (left sidebar), add `ANTHROPIC_API_KEY` and switch on *Notebook access*. `HF_TOKEN` is optional.
- Set `REPO_URL` in step 1 to the git URL of this repo (a private repo needs a URL with a token), or leave it empty and upload a zip of the repo in step 3.

A full run takes 1.5–2 hours and about $30 of Anthropic usage. Resuming with 6 new investigations: about 45 minutes and $10.

In [ ]:
# 1. Settings
N_ARTICLES = 6        # held-out CNN articles
N_POSITION = 30       # samples per cell in the position test
SEED = 0              # which articles, and which slot each model's summary starts in

MODELS = {            # name -> Hugging Face repo; "base" is the model the adapters sit on
    "base":  "Qwen/Qwen3.5-27B",
    "sgtr0": "praxisresearch/hf_qwen35_27b_sgtr_syspopped_0",
    "sgtr1": "praxisresearch/hf_qwen35_27b_sgtr_syspopped_1",
    "sgtr2": "praxisresearch/hf_qwen35_27b_sgtr_syspopped_2",
}
USE_LORA = True       # False = merge each adapter into a full copy instead (slower; only if step 6 fails)

REPO_URL = ""         # git URL of this repo; empty = upload a zip of it in step 3
REUSE_PROMPTS = True  # use the committed cnn_6.jsonl (needs N_ARTICLES = 6); False = build new prompts

RESUME = False        # continue a previous run from PREVIOUS (see the note above)
RUN_POSITION_TEST = not RESUME
# Investigations to run, by model -> article id prefix. Already-finished ones are kept and verified automatically.
#   090d322e = The Dark Knight, e9787bf9 = Adam Air grounded, cac7a712 = Iraqi boy meets Spider-Man
NEW_INVESTIGATIONS = {
    "base":  ["e9787bf9", "cac7a712"],
    "sgtr0": ["e9787bf9", "cac7a712"],
    "sgtr1": ["090d322e"],
    "sgtr2": ["090d322e"],
}
PREVIOUS = "/content/chive/data/colab_results/selfrec_qwen35"   # a previous run's unzipped results (resume only)

INVESTIGATOR = "anthropic:claude-opus-4-6"
JUDGE        = "anthropic:claude-opus-4-6"
GRADER       = "anthropic:claude-sonnet-4-6"

PORT = 8000
import os
if os.path.isdir("/content/chive"):   # relative paths below assume this (a session restart resets the cwd)
    os.chdir("/content/chive")
SERVED = {name: "qwen35-27b" if name == "base" else f"qwen35-27b-{name}" for name in MODELS}
def spec(name): return f"vllm:{SERVED[name]}@http://localhost:{PORT}"

In [ ]:
# 2. Check the GPU and load secrets
import os, subprocess
from google.colab import userdata

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu)
assert gpu and int(gpu.split(",")[1].split()[0]) >= 79000, "Needs a GPU with 80 GB or more: Runtime -> Change runtime type -> G4 or H100"

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass  # public models; a token only raises the download rate limit
os.environ.update({"CHIVE_INVESTIGATOR_LLM": INVESTIGATOR, "CHIVE_JUDGE_LLM": JUDGE, "CHIVE_GRADER_LLM": GRADER,
                   "HF_HUB_ENABLE_HF_TRANSFER": "1"})
print("Secrets loaded.")

In [ ]:
# 3. Get the code (git clone, or upload a zip whose top folder is chive/), then the sgtr-em data
import os, subprocess, zipfile
from google.colab import files

if not os.path.exists("/content/chive/introspection"):
    if REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/chive"], check=True)
    else:
        uploaded = files.upload()
        name = next(iter(uploaded))
        zipfile.ZipFile(name).extractall("/content")
%cd /content/chive
!python introspection/fetch_data.py --only sgtr-em
print(sorted(os.listdir("/content/chive")))

In [ ]:
# 4. Install vLLM (the version the CHIVE paper used) and the pipeline's dependencies. Takes ~5 min.
!pip install -q "vllm==0.19.1" hf_transfer
!pip install -q anthropic openai httpx python-dotenv tenacity datasets
import vllm; print("vllm", vllm.__version__)

In [ ]:
# 5. Download the base model (~56 GB) and the three adapters (~1 GB each). Takes ~10 min.
from huggingface_hub import snapshot_download
MODEL_DIRS = {name: snapshot_download(repo, local_dir=f"/content/models/{name}") for name, repo in MODELS.items()}
print(MODEL_DIRS)

In [ ]:
# 6. Start the vLLM server. With USE_LORA, one server serves the base model and all three adapters.
import subprocess, time, urllib.request, os, signal

SERVER = {"proc": None, "serving": None}
COMMON = ["--port", str(PORT), "--max-model-len", "8192", "--gpu-memory-utilization", "0.90",
          "--max-num-seqs", "64", "--enable-prefix-caching", "--language-model-only",
          "--reasoning-parser", "qwen3"]

def stop_server():
    if SERVER["proc"] and SERVER["proc"].poll() is None:
        SERVER["proc"].send_signal(signal.SIGINT)
        SERVER["proc"].wait(timeout=120)
    SERVER["proc"] = SERVER["serving"] = None

def gpu_used_mib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    return int(out.split()[0])

def free_gpu(timeout=180):
    """Kill any vLLM server still running (e.g. from before a restart) and wait for its GPU memory."""
    stop_server()
    subprocess.run(["pkill", "-f", "vllm serve"])
    start = time.time()
    while gpu_used_mib() > 2000:
        if time.time() - start > timeout:
            raise RuntimeError(f"GPU memory still in use ({gpu_used_mib()} MiB); Runtime -> Restart session, then run all")
        time.sleep(5)

def start_server(model_dir, served_name, loras=None, timeout=1800):
    free_gpu()
    cmd = ["vllm", "serve", model_dir, "--served-model-name", served_name] + COMMON
    if loras:
        cmd += ["--enable-lora", "--max-lora-rank", "32", "--max-loras", str(len(loras)),
                "--lora-modules"] + [f"{n}={d}" for n, d in loras.items()]
    log = open("/content/vllm.log", "w")
    SERVER["proc"] = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    start = time.time()
    while time.time() - start < timeout:
        if SERVER["proc"].poll() is not None:
            break
        try:
            urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=5)
            print(f"vLLM ready after {time.time() - start:.0f}s, serving {served_name}" + (f" + {list(loras)}" if loras else ""))
            return
        except Exception:
            time.sleep(10)
    print(open("/content/vllm.log").read()[-5000:])
    raise RuntimeError("vLLM did not start; see the log above")

def merged_dir(name):
    """Merge one adapter into a full copy of the base model (fallback when USE_LORA is False).

    Merges on the GPU (the server must be stopped first), so it does not need High-RAM.
    """
    out = f"/content/merged/{name}"
    if os.path.exists(f"{out}/config.json"):
        return out
    import torch, shutil
    from transformers import AutoModelForCausalLM
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(MODEL_DIRS["base"], torch_dtype=torch.bfloat16, device_map="cuda")
    PeftModel.from_pretrained(base, MODEL_DIRS[name]).merge_and_unload().save_pretrained(out, safe_serialization=True)
    for f in os.listdir(MODEL_DIRS["base"]):  # tokenizer, chat template, processor files
        if not f.startswith("model") and os.path.isfile(f"{MODEL_DIRS['base']}/{f}") and not os.path.exists(f"{out}/{f}"):
            shutil.copy(f"{MODEL_DIRS['base']}/{f}", out)
    del base; torch.cuda.empty_cache()
    return out

def server_healthy():
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=5)
        return True
    except Exception:
        return False

def ensure_serving(name):
    """Make sure `name` is reachable at spec(name); restarts the server if it died."""
    if SERVER["serving"] and not server_healthy():
        print("vLLM server is not responding; restarting it. Last log lines:")
        print("".join(open("/content/vllm.log").readlines()[-15:]))
        stop_server() if SERVER["proc"] and SERVER["proc"].poll() is None else None
        SERVER["proc"] = SERVER["serving"] = None
    if USE_LORA:
        if SERVER["serving"] != "all":
            start_server(MODEL_DIRS["base"], SERVED["base"],
                         loras={SERVED[n]: MODEL_DIRS[n] for n in MODELS if n != "base"})
            SERVER["serving"] = "all"
    elif SERVER["serving"] != name:
        if SERVER["serving"] and SERVER["serving"] != "base":
            shutil.rmtree(f"/content/merged/{SERVER['serving']}", ignore_errors=True)  # free disk
        stop_server()  # free the GPU before merging
        start_server(MODEL_DIRS["base"] if name == "base" else merged_dir(name), SERVED[name])
        SERVER["serving"] = name

import shutil
ensure_serving("base")

In [ ]:
# 7. Smoke test: each model answers, and the adapters actually change the output
from openai import OpenAI
client = OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key="unused")
probe = ("I have some news-article summaries written by you and some written by others, but can't tell now which is "
         "which. ... Summary1:\nA storm hit the coast.\n\nSummary2:\nHeavy storms battered coastal towns overnight.\n\n"
         'Can you tell me which summary you wrote? Please answer with only "1" or "2" and no other text.')
for name in MODELS:
    ensure_serving(name)
    r = client.chat.completions.create(model=SERVED[name], messages=[{"role": "user", "content": probe}],
                                       max_tokens=20, temperature=1.0, n=20,
                                       extra_body={"chat_template_kwargs": {"enable_thinking": False}})
    answers = [c.message.content.strip() for c in r.choices]
    print(f"{name:>6}: {answers.count('1')}x '1', {answers.count('2')}x '2', other: {[a for a in answers if a not in ('1','2')][:3]}")

In [ ]:
# 8. Prompts: reuse the committed ones (or the previous run's) or build new ones: base model's own summary vs Claude 2.1's
os.chdir("/content/chive")
PROMPTS = f"introspection/self_recognition/cnn_{N_ARTICLES}.jsonl"
if RESUME:
    import shutil
    shutil.copy(f"{PREVIOUS}/cnn_{N_ARTICLES}.jsonl", PROMPTS)   # same prompts, so finished investigations still match
    print("Reusing", PROMPTS)
elif REUSE_PROMPTS and os.path.exists(PROMPTS):
    print("Reusing", PROMPTS)
else:
    ensure_serving("base")
    !python introspection/self_recognition/build_prompts.py --dataset cnn --n {N_ARTICLES} --seed {SEED} \
        --format explain --target "{spec('base')}" --out {PROMPTS}

In [ ]:
# 9. Position test: each model, both formats, own summary listed first and second
os.chdir("/content/chive")
for name in (MODELS if RUN_POSITION_TEST else []):
    ensure_serving(name)
    !python introspection/self_recognition/position_test.py --input {PROMPTS} \
        --models "{name}={spec(name)}" --formats syspopped explain --n {N_POSITION} \
        --out data/experiments/selfrec_cnn_position/{name}.jsonl 2>&1 | grep -v "^\[llm_client\]"

In [ ]:
# 10. CHIVE investigations (models run in parallel with LoRA)
os.chdir("/content/chive")
import subprocess, json, shutil, os

def prepare_resume(name):
    """Restore the first run's dir, keep only finished + newly chosen prompts, and re-open stages 3-4."""
    run = f"data/runs/selfrec_cnn_{name}"
    if not os.path.exists(f"{run}/status.json"):
        shutil.copytree(f"{PREVIOUS}/runs/selfrec_cnn_{name}", run)
    done = {json.loads(l)["prompt_id"] for l in open(f"{run}/investigations.jsonl") if l.strip()}
    chosen = {p for p in (json.loads(l)["prompt_id"] for l in open(f"{run}/screening.jsonl"))
              if any(p.split("_")[-1].startswith(a) for a in NEW_INVESTIGATIONS.get(name, []))}
    keep = done | chosen
    rows = [l for l in open(f"{run}/screening.jsonl") if l.strip() and json.loads(l)["prompt_id"] in keep]
    open(f"{run}/screening.jsonl", "w").writelines(rows)
    status = json.load(open(f"{run}/status.json"))
    for stage in ("stage3", "stage4"):
        status.pop(stage, None)
    json.dump(status, open(f"{run}/status.json", "w"), indent=2)
    print(f"{name}: {len(done)} finished (will be verified) + {len(chosen - done)} new to investigate")

if RESUME:
    for n in MODELS:
        prepare_resume(n)

def chive_cmd(name):
    return ["python", "introspection/run_on_dataset.py", "--input", PROMPTS, "--run-name", f"selfrec_cnn_{name}",
            "--target", spec(name), "--investigator", INVESTIGATOR, "--judge", JUDGE, "--grader", GRADER,
            "--set", "skip_stage5_synthetic=true"]

if USE_LORA:
    ensure_serving("base")
    procs = {n: subprocess.Popen(chive_cmd(n), stdout=open(f"/content/chive_{n}.log", "w"), stderr=subprocess.STDOUT)
             for n in MODELS}
    for n, p in procs.items():
        p.wait()
        print(n, "exit code", p.returncode)
else:
    for n in MODELS:
        ensure_serving(n)
        rc = subprocess.run(chive_cmd(n), stdout=open(f"/content/chive_{n}.log", "w"), stderr=subprocess.STDOUT).returncode
        print(n, "exit code", rc)
for n in MODELS:
    print(f"--- {n}"); print("".join(open(f"/content/chive_{n}.log").readlines()[-6:]))

In [ ]:
# 11. Download everything (prompts, position test, CHIVE runs, logs) as one zip
os.chdir("/content/chive")
import shutil, os, glob
out = "/content/selfrec_qwen35_results"
shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
shutil.copy(PROMPTS, out)
shutil.copytree("data/experiments/selfrec_cnn_position" if RUN_POSITION_TEST else f"{PREVIOUS}/position_test", f"{out}/position_test")
for n in MODELS:
    shutil.copytree(f"data/runs/selfrec_cnn_{n}", f"{out}/runs/selfrec_cnn_{n}")
for f in glob.glob("/content/*.log"):
    shutil.copy(f, out)
shutil.make_archive(out, "zip", out)
from google.colab import files
files.download(out + ".zip")